# MIT License

Copyright (c) 2024 Social Science Research Notebooks

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

In [ ]:
# Setup for Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print('Running in Colab')


# Hate Speech / Toxicity Detection In-Depth

This notebook demonstrates a supervised learning pipeline for detecting toxic comments. We will generate a synthetic dataset, extract features using TF-IDF, train a Logistic Regression model, and evaluating its performance using industry-standard metrics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

sns.set_style('whitegrid')

## 1. Generate Synthetic Dataset
We simulate comments found on social media platforms, categorized as 'Toxic' or 'Clean'. In a real scenario, you would use datasets like the Wikipedia Talk Labels or Twitter Hate Speech datasets.

In [ ]:
clean_comments = [
    "I love this video, thanks for sharing!",
    "Great explanation, very clear.",
    "Have a wonderful day everyone.",
    "I disagree with your point, but I respect your opinion.",
    "Can you make a tutorial on Python?",
    "This is exactly what I was looking for.",
    "Keep up the good work.",
    "Interesting perspective.",
    "The audio quality is excellent.",
    "Hello from Brazil!"
]

toxic_comments = [
    "You are an idiot and you know nothing.",
    "Go away, nobody likes you.",
    "This is the worst video I have ever seen.",
    "You are so stupid, delete your channel.",
    "I hate you and your content.",
    "Shut up, you loser.",
    "Garbage content for garbage people.",
    "You are ugly and dumb.",
    "Stop posting this trash.",
    "You are a waste of space."
]

# Create larger dataset by repeating and shuffling
np.random.seed(42)
X = clean_comments * 10 + toxic_comments * 10
y = [0] * 100 + [1] * 100 # 0 = Clean, 1 = Toxic

df = pd.DataFrame({'text': X, 'label': y})
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Dataset Size: {len(df)}")
print(df['label'].value_counts())

## 2. Train-Test Split
It is crucial to hold out a test set to evaluate how the model performs on unseen data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)
print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

## 3. Feature Extraction (TF-IDF)
We convert text to numbers using TF-IDF (Term Frequency-Inverse Document Frequency), which highlights words that are unique to specific documents.

In [ ]:
vectorizer = TfidfVectorizer(stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

## 4. Model Training
Logistic Regression is a strong baseline for binary text classification.

In [ ]:
model = LogisticRegression()
model.fit(X_train_vec, y_train)
print("Model trained.")

## 5. Evaluation
We check Accuracy, Precision (how many predicted toxic were actually toxic), and Recall (how many actual toxic comments we caught).

In [ ]:
y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=['Clean', 'Toxic']))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', xticklabels=['Clean', 'Toxic'], yticklabels=['Clean', 'Toxic'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix')
plt.show()

## 6. Test on New Examples
Let's try some custom phrases.

In [ ]:
new_comments = ["You are amazing!", "You are a terrible person.", "I don't agree, but nice video."]
new_vec = vectorizer.transform(new_comments)
preds = model.predict(new_vec)

for comment, pred in zip(new_comments, preds):
    label = "Toxic" if pred == 1 else "Clean"
    print(f"'{comment}' -> {label}")